# Análise do Dataset e Treinamento do Modelo

Notebook de análise exploratoria e treinamento do modelo usando o dataset **UCI HAR - Human Activity Recognition Using Smartphones**.

Fluxo seguido neste notebook:

1. Importar bibliotecas
2. Carregar o dataset
3. Analisar estrutura dos dados
4. Analisar distribuição das classes
5. Visualizar sinais dos sensores
6. Preparar features para treinamento
7. Treinar modelos
8. Comparar resultados
9. Avaliar o modelo escolhido
10. Exportar o modelo compacto para uso embarcado


## 0. Preparação do ambiente

Execute esta célula antes dos imports. Ela instala, no kernel atual do notebook, as bibliotecas necessárias para a análise e o treinamento.

Isso evita o erro `ModuleNotFoundError`, como o erro de `matplotlib` que apareceu no VS Code.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

matplotlib_cache = Path.cwd() / '.matplotlib-cache'
matplotlib_cache.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(matplotlib_cache)

packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    print('Instalando dependências ausentes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('Todas as dependências já estão instaladas neste kernel.')


## 1. Imports

Aqui importamos as bibliotecas usadas para carregar dados, analisar, visualizar e treinar modelos.


In [ ]:
from pathlib import Path
import os

matplotlib_cache = Path.cwd() / '.matplotlib-cache'
matplotlib_cache.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(matplotlib_cache)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

%matplotlib inline

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')


## 2. Configurações do projeto

Definimos caminhos, nomes das classes e nomes dos sinais de sensor que vamos usar.


In [ ]:
current_dir = Path.cwd()

if (current_dir / 'data').exists() or (current_dir / 'scripts').exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

DATA_ROOT = PROJECT_ROOT / 'data' / 'uci_har'

ACTIVITY_NAMES = [
    'WALKING',
    'WALKING_UPSTAIRS',
    'WALKING_DOWNSTAIRS',
    'SITTING',
    'STANDING',
    'LAYING',
]

SENSOR_FILES = [
    'total_acc_x',
    'total_acc_y',
    'total_acc_z',
    'body_gyro_x',
    'body_gyro_y',
    'body_gyro_z',
]

RANDOM_STATE = 42

print('Raiz do projeto:', PROJECT_ROOT)
print('Pasta do dataset:', DATA_ROOT)


## 3. Localizando o dataset

O dataset precisa ter sido baixado antes com:

```bash
python scripts/download_dataset.py
```


In [ ]:
def find_dataset_base():
    matches = list(DATA_ROOT.rglob('UCI HAR Dataset'))
    for match in matches:
        if (match / 'train' / 'Inertial Signals').exists() and (match / 'test' / 'Inertial Signals').exists():
            return match
    raise FileNotFoundError('Dataset nao encontrado. Rode: python scripts/download_dataset.py')

dataset_base = find_dataset_base()
dataset_base


## 4. Carregando rótulos e sinais inerciais

Vamos carregar os sinais crus de acelerômetro e giroscópio.

Cada arquivo de sinal tem o formato:

- linhas = janelas de movimento
- colunas = 128 leituras dentro daquela janela


In [ ]:
def load_signal(split, sensor_name):
    path = dataset_base / split / 'Inertial Signals' / f'{sensor_name}_{split}.txt'
    return np.loadtxt(path)

def load_all_signals(split):
    return {sensor_name: load_signal(split, sensor_name) for sensor_name in SENSOR_FILES}

def load_labels(split):
    path = dataset_base / split / f'y_{split}.txt'
    return np.loadtxt(path, dtype=np.int64) - 1

train_signals = load_all_signals('train')
test_signals = load_all_signals('test')
y_train = load_labels('train')
y_test = load_labels('test')

print('Treino:', len(y_train), 'janelas')
print('Teste:', len(y_test), 'janelas')
print('Formato de um sinal:', train_signals['total_acc_x'].shape)


## 5. Análise inicial da estrutura

Nestá etapa conferimos se todos os sinais possuem a mesma quantidade de linhas e colunas.


In [ ]:
signal_summary = []

for name, values in train_signals.items():
    signal_summary.append({
        'sinal': name,
        'janelas_treino': values.shape[0],
        'leituras_por_janela': values.shape[1],
        'media_geral': values.mean(),
        'desvio_geral': values.std(),
        'minimo': values.min(),
        'maximo': values.max(),
    })

pd.DataFrame(signal_summary)


## 6. Análise da distribuição das classes

Antes de treinar, verificamos se as classes estáo equilibradas.

Se uma classe tiver muitos exemplos e outra tiver poucos, o modelo pode ficar enviesado.


In [ ]:
def class_distribution(labels, split_name):
    counts = np.bincount(labels, minlength=len(ACTIVITY_NAMES))
    return pd.DataFrame({
        'split': split_name,
        'classe': ACTIVITY_NAMES,
        'quantidade': counts,
        'percentual': counts / counts.sum()
    })

dist_train = class_distribution(y_train, 'treino')
dist_test = class_distribution(y_test, 'teste')
dist = pd.concat([dist_train, dist_test], ignore_index=True)
dist


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

x = np.arange(len(ACTIVITY_NAMES))
width = 0.35

ax.bar(x - width / 2, dist_train['quantidade'], width, label='treino')
ax.bar(x + width / 2, dist_test['quantidade'], width, label='teste')
ax.set_xticks(x)
ax.set_xticklabels(ACTIVITY_NAMES, rotation=30, ha='right')
ax.set_ylabel('Quantidade de janelas')
ax.set_title('Distribuicao das classes')
ax.legend()
plt.tight_layout()
plt.show()


## 7. Visualização de sinais do sensor

Agora visualizamos exemplos reais das janelas.

Isso ajuda a perceber que atividades diferentes geram padrões diferentes nos sensores.


In [ ]:
def first_index_of_class(labels, class_index):
    return int(np.where(labels == class_index)[0][0])

fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
axes = axes.ravel()

for class_index, ax in enumerate(axes):
    idx = first_index_of_class(y_train, class_index)
    ax.plot(train_signals['total_acc_x'][idx], label='acc_x')
    ax.plot(train_signals['total_acc_y'][idx], label='acc_y')
    ax.plot(train_signals['total_acc_z'][idx], label='acc_z')
    ax.set_title(ACTIVITY_NAMES[class_index])
    ax.set_xlabel('leitura dentro da janela')
    ax.set_ylabel('aceleracao')

axes[0].legend()
plt.tight_layout()
plt.show()


## 8. Feature engineering

Para embarcar em um ESP32-S3, queremos uma entrada pequena.

Por isso, para cada sinal calculamos:

- média
- desvio padrao
- mínimo
- máximo
- RMS

Como são 6 sinais e 5 medidas, ficamos com 30 features por janela.


In [ ]:
FEATURE_NAMES = []
for sensor_name in SENSOR_FILES:
    FEATURE_NAMES.extend([
        f'{sensor_name}_mean',
        f'{sensor_name}_std',
        f'{sensor_name}_min',
        f'{sensor_name}_max',
        f'{sensor_name}_rms',
    ])

def extract_features(signals_dict):
    features = []
    for sensor_name in SENSOR_FILES:
        signal = signals_dict[sensor_name]
        features.append(signal.mean(axis=1))
        features.append(signal.std(axis=1))
        features.append(signal.min(axis=1))
        features.append(signal.max(axis=1))
        features.append(np.sqrt((signal * signal).mean(axis=1)))
    return np.column_stack(features)

X_train = extract_features(train_signals)
X_test = extract_features(test_signals)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)


## 9. Tabela final de treino

Aqui criamos um DataFrame para analisar estatísticas das features criadas.


In [ ]:
df_train = pd.DataFrame(X_train, columns=FEATURE_NAMES)
df_train['classe'] = [ACTIVITY_NAMES[i] for i in y_train]

df_train.head()


In [ ]:
df_train.describe().T.head(15)


## 10. Comparando classes por uma feature

Um passo comum em EDA e verificar se algumas features mudam conforme a classe.

Abaixo olhamos o desvio padrao da aceleracao no eixo X por atividade.


In [ ]:
feature_to_plot = 'total_acc_x_std'

fig, ax = plt.subplots(figsize=(10, 4))
df_train.boxplot(column=feature_to_plot, by='classe', ax=ax, rot=30)
ax.set_title(f'Distribuicao de {feature_to_plot} por classe')
ax.set_ylabel(feature_to_plot)
plt.suptitle('')
plt.tight_layout()
plt.show()


## 11. Baseline

Antes de treinar o modelo real, criamos um baseline.

O baseline e um modelo bobo que sempre tenta chutar a classe mais frequente.

O nosso modelo real precisa ser melhor que isso.


In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_acc


## 12. Treinamento de modelos

Nesta etapa treinamos três modelos:

1. baseline: chute simples da classe mais frequente.
2. Árvore de decisão compacta: modelo simples usado como comparação.
3. MLP compacta: modelo final escolhido para embarcar.

A MLP foi escolhida porque melhorou a acurácia e continuou pequena o suficiente para rodar no ESP32-S3.


In [ ]:
modelo_arvore = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=8,
    random_state=RANDOM_STATE,
)

modelo_mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(16,),
        activation='relu',
        solver='adam',
        alpha=0.0005,
        max_iter=800,
        early_stopping=True,
        n_iter_no_change=25,
        random_state=RANDOM_STATE,
        learning_rate_init=0.001,
    ),
)

modelo_arvore.fit(X_train, y_train)
modelo_mlp.fit(X_train, y_train)

mlp = modelo_mlp.named_steps['mlpclassifier']
mlp_params = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)

print('Nós da Árvore compacta:', modelo_arvore.tree_.node_count)
print('Parâmetros da MLP compacta:', mlp_params)


## 13. Validação cruzada no treino

A validação cruzada mede o desempenho do modelo em diferentes partes do conjunto de treino.

Aqui validamos a MLP compacta, que é o modelo final do projeto.


In [ ]:
cv_scores = cross_val_score(modelo_mlp, X_train, y_train, cv=5, scoring='accuracy')

print('Acurácias da validação cruzada:', np.round(cv_scores, 4))
print('Média:', cv_scores.mean())
print('Desvio:', cv_scores.std())


## 14. Avaliação no conjunto de teste

Agora avaliamos com dados que o modelo não usou no treinamento.


In [ ]:
pred_arvore = modelo_arvore.predict(X_test)
pred_mlp = modelo_mlp.predict(X_test)

results = pd.DataFrame([
    {
        'modelo': 'baseline',
        'acuracia_teste': baseline_acc,
        'tamanho': 'sem modelo real',
    },
    {
        'modelo': 'arvore_compacta',
        'acuracia_teste': accuracy_score(y_test, pred_arvore),
        'tamanho': f'{modelo_arvore.tree_.node_count} nós',
    },
    {
        'modelo': 'mlp_compacta_final',
        'acuracia_teste': accuracy_score(y_test, pred_mlp),
        'tamanho': f'{mlp_params} parâmetros',
    },
])

results


## 15. Relatório de classificação

O relatorio mostra precision, recall e f1-score para cada classe.


In [ ]:
print(classification_report(y_test, pred_mlp, target_names=ACTIVITY_NAMES))


## 16. Matriz de confusão

A matriz de confusão mostra em quais classes o modelo acerta e em quais ele confunde.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_mlp,
    display_labels=ACTIVITY_NAMES,
    xticks_rotation=45,
    cmap='Blues',
    ax=ax,
)
ax.set_title('Matriz de confusão - MLP compacta')
plt.tight_layout()
plt.show()


## 17. Comparação visual dos modelos

Aqui comparamos o baseline, a Árvore compacta e a MLP compacta final.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(results['modelo'], results['acuracia_teste'], color=['gray', '#F28E2B', '#1B9E77'])
ax.set_ylim(0, 1)
ax.set_ylabel('Acurácia no teste')
ax.set_title('Comparação de acurácia')
for index, row in results.iterrows():
    ax.text(index, row['acuracia_teste'] + 0.02, f"{row['acuracia_teste']:.2%}", ha='center')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
results


## 18. Como a MLP funciona

A MLP recebe as 30 características normalizadas, calcula uma camada oculta com 16 neurônios usando ReLU e depois calcula 6 pontuações, uma para cada classe.

A classe prevista é aquela que fica com a maior pontuação.


In [ ]:
print('Entradas:', X_train.shape[1])
print('Camada oculta:', mlp.hidden_layer_sizes[0], 'neurônios')
print('Saídas:', len(ACTIVITY_NAMES), 'classes')
print('Formato dos pesos da entrada para a camada oculta:', mlp.coefs_[0].shape)
print('Formato dos pesos da camada oculta para a saída:', mlp.coefs_[1].shape)


## 19. Testando uma amostra individual

Aqui pegamos uma janela do conjunto de teste e verificamos a predição do modelo.


In [ ]:
sample_index = 0
sample_features = X_test[sample_index]
real_class = y_test[sample_index]
predicted_class = modelo_mlp.predict([sample_features])[0]
probabilities = modelo_mlp.predict_proba([sample_features])[0]

print('Classe real:', ACTIVITY_NAMES[real_class])
print('Classe prevista:', ACTIVITY_NAMES[predicted_class])
print()
print('Probabilidades por classe:')
pd.DataFrame({
    'classe': ACTIVITY_NAMES,
    'probabilidade': probabilities,
}).sort_values('probabilidade', ascending=False)


## 20. Exportação do modelo para embarcado

O treino oficial do projeto é feito pelo script:

```bash
python scripts/train_and_export.py
```

Esse script repete a preparação das features, treina a MLP compacta e exporta o modelo para C/C++.

O arquivo exportado fica em:

```text
firmware/wokwi/model_data.h
```

No Wokwi, o ESP32-S3 usa esse arquivo para fazer a inferência.


In [ ]:
model_header = PROJECT_ROOT / 'firmware' / 'wokwi' / 'model_data.h'
print('Arquivo exportado existe?', model_header.exists())
print('Caminho:', model_header)

if model_header.exists():
    for line in model_header.read_text(encoding='utf-8').splitlines()[:20]:
        print(line)


## 21. Conclusão da análise

Conclusões principais:

- O dataset possui sinais de acelerômetro e giroscópio separados em janelas.
- Cada janela foi resumida em 30 features estatísticas.
- A MLP compacta teve desempenho melhor que a Árvore compacta.
- O modelo final ficou pequeno, com 598 parâmetros.
- A MLP foi exportada para C/C++ para rodar no ESP32-S3 simulado no Wokwi.
